# Multi-Agent Deep Q-Network (DQN) Coordination in a Grid World

## Project Overview

This project implements a **Multi-Agent Deep Q-Network (DQN)** to solve a cooperative transportation task in a **5×5 Grid World**. Four autonomous agents repeatedly transport items between a pickup location (**A**) and a drop-off location (**B**) while learning to coordinate their movements and avoid head-on collisions.

Unlike traditional single-agent reinforcement learning, this project focuses on **multi-agent coordination**, where each agent must make decisions that not only maximize its own reward but also contribute to efficient group behavior.

The project is implemented **from scratch** without using reinforcement learning libraries. PyTorch (or TensorFlow) is used only to build the Deep Q-Network.

---

# Objectives

The primary objectives of this project are:

- Build a custom 5×5 Grid World environment.
- Simulate four independent agents.
- Train agents using the Deep Q-Network (DQN) algorithm.
- Learn efficient transportation between pickup and drop-off locations.
- Minimize head-on collisions through learned coordination.
- Evaluate the trained agents on unseen scenarios.

---

# Problem Description

Each agent continuously performs the following cycle:

1. Move toward the pickup location (**A**).
2. Pick up an item automatically.
3. Move toward the drop-off location (**B**).
4. Deliver the item automatically.
5. Return to the pickup location.
6. Repeat the process.

Agents move simultaneously (executed sequentially in random order each timestep) and must learn to coordinate their actions to avoid collisions while maintaining efficient deliveries.

---

# Environment Specifications

| Property | Value |
|-----------|-------|
| Grid Size | 5 × 5 |
| Number of Agents | 4 |
| Pickup Location | A |
| Drop-off Location | B |
| Action Space | North, South, East, West |
| Wait Action | Not Allowed |
| Movement Order | Random Sequential |
| Learning Algorithm | Deep Q-Network (DQN) |

---

# Observation Space

Each agent observes:

- Its current position
- Pickup location (A)
- Drop-off location (B)
- Whether it is currently carrying an item

The observation is converted into a numerical state vector before being passed to the neural network.

---

# Action Space

Each agent can perform one of four actions:

| Action | Description |
|----------|-------------|
| North | Move one cell upward |
| South | Move one cell downward |
| East | Move one cell to the right |
| West | Move one cell to the left |

No **Wait** action is allowed.

---

# Head-On Collision Rule

A collision occurs **only** when:

- One agent is travelling **from A to B**, and
- Another agent is travelling **from B to A**, and
- Both occupy the same cell after movement.

Special cases:

- Multiple agents may occupy the same cell if they are travelling in the same direction.
- Collisions occurring on the pickup location (A) or drop-off location (B) are ignored.

---

# Reinforcement Learning Approach

This project uses the **Deep Q-Network (DQN)** algorithm.

Unlike Tabular Q-Learning, which stores Q-values inside a lookup table, DQN approximates the action-value function using a neural network.

The network predicts the expected future reward for every possible action given the current state.

---

# DQN Learning Equation

The target Q-value is computed using

$$Q(s_t,a_t)=r_t+\gamma\max_aQ(s_{t+1},a)$$

The neural network is trained to minimize the difference between the predicted Q-value and the target Q-value.

---

# Project Components

The implementation is divided into several stages:

1. Environment Construction
2. Agent Representation
3. Collision Detection
4. State Representation
5. Deep Q-Network
6. DQN Agent
7. Replay buffer
8. Training Loop
9. Performance Evaluation
10. Visualization

---

# Training Constraints

The implementation follows the assignment requirements.

| Constraint | Limit |
|------------|-------|
| Maximum Training Steps | 1,500,000 |
| Maximum Collisions | 4,000 |
| Maximum Training Time | 10 minutes |

---

# Performance Goal

After training, the system should:

- Complete a full delivery cycle successfully.
- Finish within **25 steps**.
- Remain collision-free.
- Achieve at least **75% success rate** over multiple evaluation scenarios.

---

## Import Libraries

#### Imports use cases

| Import | Used For |
|--------|----------|
| `random` | ε-greedy exploration (choosing random actions during training) |
| `deque` | Replay Buffer (efficiently storing and removing experiences) |
| `namedtuple` | Experience storage (`state`, `action`, `reward`, `next_state`, `done`) |
| `dataclass` | Organizing hyperparameters and configuration values |
| `numpy` | State representation, numerical operations, and data processing |
| `torch` | Building and training deep learning models |
| `torch.nn (nn)` | Defining the Q-Network architecture (layers, modules) |
| `torch.nn.functional (F)` | Activation functions and loss functions (e.g., ReLU, MSE Loss) |
| `matplotlib` | Visualizing training progress (rewards, losses, success rate) |
| `IntEnum` | Creating readable action definitions (e.g., `UP`, `DOWN`, `LEFT`, `RIGHT`) |

In [6]:
# --- Imports ---
import math, os, time, random
from enum import IntEnum
from collections import deque, namedtuple
from dataclasses import dataclass
from typing import List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cpu")
print("Torch:", torch.__version__, "| Device:", device)

Torch: 2.13.0+cpu | Device: cpu


## 1. Actions (Enum) and environment constants

In [7]:
class Action(IntEnum):
    NORTH = 0  # y - 1 (UP)
    SOUTH = 1  # y + 1 (DOWN)
    EAST  = 2  # x + 1 (RIGHT)
    WEST  = 3  # x - 1 

# Unit vector for each action (dx, dy)
ACTION_DELTA = {
    Action.NORTH: (0, -1),
    Action.SOUTH: (0,  1),
    Action.EAST:  (1,  0),
    Action.WEST:  (-1, 0),
}
N_ACTIONS = 4

GRID = 5
N_AGENTS = 4

@dataclass(frozen=True)
class Position:
    x: int
    y: int

A_POS = Position(0, 0)   # pick-up
B_POS = Position(4, 4)   # drop-off

## 2. Environment

* The environment tracks each agent's `(x, y)` and `carrying` flag. 
* On each **tick**, we shuffle the agent order, then step each agent once. 
* After each individual move, we check whether the new cell (which is neither `A` nor `B`) contains at least one agent going in the *opposite* travel direction: if so, it counts as a **head-oncollision.** 
* The environment reports how many collisions happened during a tick and which agents were involved.

**Travel direction** is derived from the `carrying` flag:
- `carrying == True` → the agent is moving `A → B`.
- `carrying == False` → the agent is moving `B → A`.

#### Responsibilities of each functions
| Function        | Responsibility                                                            |
| --------------- | ------------------------------------------------------------------------- |
| `__init__()`    | Create the environment                                                    |
| `reset()`       | Start a new episode                                                       |
| `obs()`         | Build one agent's observation                                             |
| `observe_all()` | Build observations for every agent                                        |
| `_apply_move()` | Move one agent inside the grid                                            |
| `step()`        | Execute one environment step (movement, rewards, collisions, termination) |


#### Panulty / Rewards
| Event                 |                         Reward/Penalty | Purpose                                                                       |
| --------------------- | -------------------------------------: | ----------------------------------------------------------------------------- |
| **Progress Reward**   | `+0.3 × (old_distance - new_distance)` | Reward moving closer to the current target (A or B).                          |
| **Move Penalty**      |                     `-0.05` every step | Encourages shorter paths and discourages wandering.                           |
| **Pickup Reward**     |                                 `+5.0` | Reward for successfully picking up an item at A.                              |
| **Drop-off Reward**   |                                `+10.0` | Reward for successfully delivering an item to B.                              |
| **Collision Penalty** |                                `-15.0` | Strongly discourages collisions with agents moving in the opposite direction. |



In [8]:
# AgentState dataclass
@dataclass
class AgentState:
    x: int
    y: int
    carrying: bool  # True => heading A->B, False => heading B->A

    @property
    def direction(self) -> int:
        return 1 if self.carrying else -1  # +1 = A->B, -1 = B->A
    
OBS_DIM = 10  # own(2) + A(2) + B(2) + carrying(1) + nearest-opposite dx,dy,present (3)

class GridWorld:
    def __init__(
        self,
        grid_size: int = GRID,
        pickup_pos: Position = A_POS,
        drop_pos: Position = B_POS,
        max_ticks: int = 200,
        n_agents: int = N_AGENTS,
        n_actions: int = N_ACTIONS
    ):
        self.grid_size = grid_size
        self.pick_pos = pickup_pos
        self.drop_pos = drop_pos
        self.max_ticks = max_ticks
        self.n_agents = n_agents
        self.n_actions = n_actions
        self.reset()

    #---------------------- RESET -------------------------------
    def reset(self):
        self.agents = self.create_agents()
        self.ticks = 0
        self.total_deliveries = 0
        return self.observe_all()
    
    def create_agents(self) -> List[AgentState]:
        agents = []
        for _ in range(N_AGENTS):
            position = self.random_start_position()
            agents.append(AgentState(*position, False))
        return agents
    
    def random_start_position(self) -> Tuple[int, int]:
        while True:
            position = (
                random.randrange(self.grid_size),
                random.randrange(self.grid_size),
            )

            if position in (self.pick_pos, self.drop_pos):
                continue

            return position
    
    #--------------------- OBSERVE ---------------------------------
    def obs(self, agent_id: int) -> np.ndarray:
        agent = self.agents[agent_id]
        nearest_agent = self.nearest_opposite(agent, agent_id)
        return self.build_observation(agent, nearest_agent)
    
    def nearest_opposite(
        self,
        agent: AgentState,
        agent_id: int,
    ) -> AgentState | None:
        nearest_agent = None
        best_distance = float("inf")

        for other_agent_id, other in enumerate(self.agents):  # first -> index | second -> value
            if other_agent_id == agent_id:
                continue
            if other.direction == agent.direction:
                continue

            distance = self.manhattan_distance(agent, other)
            if distance < best_distance:
                best_distance = distance
                nearest_agent = other
        return nearest_agent
    
    def manhattan_distance(
        self,
        first: AgentState,
        second: AgentState,
    ) -> int:
        return (
            abs(first.x - second.x)
            + abs(first.y - second.y)
        )
    
    def build_observation(
        self,
        agent: AgentState,
        nearest: AgentState | None,
    ) -> np.ndarray:
        opposite = self.opposite_features(agent, nearest) # (collide distance_x | collide distance_y | collide or not)
        scale = self.grid_size - 1

        return np.array([
            agent.x / scale,
            agent.y / scale,
            self.pick_pos.x / scale,
            self.pick_pos.y / scale,
            self.drop_pos.x / scale,
            self.drop_pos.y / scale,
            float(agent.carrying),
            *opposite,
        ], dtype=np.float32)
    
    def opposite_features(
        self,
        agent: AgentState,
        nearest: AgentState | None,
    ) -> Tuple[float, float, float]:
        if nearest is None:
            return (0.0, 0.0, 0.0)

        scale = self.grid_size - 1
        dx = (nearest.x - agent.x) / scale
        dy = (nearest.y - agent.y) / scale

        return (dx, dy, 1.0)
    
    def observe_all(self) -> np.ndarray:
        return np.stack(
            [self.obs(i) for i in range(self.n_agents)]
        )
    
    #-------------------------------- STEP ------------------------------------
    def step(self, actions):
        rewards = [0.0] * self.n_agents
        delivered = [False] * self.n_agents
        collided = [False] * self.n_agents

        self.process_actions(actions, rewards, delivered)
        events = self.process_collisions(rewards, collided)

        return self.finish_step(rewards, delivered, collided, events)
    
    #-----process_actions-------
    def process_actions(self, actions, rewards, delivered):
        order = list(range(self.n_agents))
        random.shuffle(order)  # shuffle agents order

        for index in order:
            self.process_agent(index, actions[index], rewards, delivered)
    
    def process_agent(self, index, action, rewards, delivered):
        agent = self.agents[index]

        old_distance = self.target_distance(agent)
        self.apply_move(index, action)
        new_distance = self.target_distance(agent)

        rewards[index] += self.progress_reward(old_distance, new_distance)
        rewards[index] -= 0.05  # move panulty

        self.handle_pickup(agent, rewards, index)
        self.handle_delivery(agent, rewards, delivered, index)

    def target_distance(self, agent):
        if agent.carrying:
            target = self.drop_pos
        else:
            target = self.pick_pos

        dx = abs(agent.x - target.x)
        dy = abs(agent.y - target.y)
        return dx + dy

    def progress_reward(self, old_distance, new_distance):
        return 0.3 * (old_distance - new_distance) 
    
    def handle_pickup(self, agent, rewards, index):
        if agent.carrying:
            return

        if (agent.x, agent.y) == (self.pick_pos.x, self.pick_pos.y):
            agent.carrying = True
            rewards[index] += 5.0

    def handle_delivery(self, agent, rewards, delivered, index):
        if agent.carrying == False:
            return

        if (agent.x, agent.y) == (self.drop_pos.x, self.drop_pos.y):
            agent.carrying = False
            rewards[index] += 10.0
            delivered[index] = True
            self.total_deliveries += 1

    def apply_move(self, index, action):
        agent = self.agents[index]
        dx, dy = ACTION_DELTA[Action(action)]

        agent.x = self.clamp(agent.x + dx)
        agent.y = self.clamp(agent.y + dy)

    def clamp(self, value):         # checks boundaries
        return min(max(value, 0), self.grid_size - 1)

    #--------- Process collisions -------
    def process_collisions(self, rewards, collided):
        cells = self.group_agents()
        events = 0

        for cell, indices in cells.items():
            events += self.check_collision(
                cell,
                indices,
                rewards,
                collided,
            )

        return events
    
    def group_agents(self):
        cells = {}

        for index, agent in enumerate(self.agents):
            position = (agent.x, agent.y)
            cells.setdefault(position, []).append(index)

        return cells
    
    def check_collision(
        self,
        cell,
        indices,
        rewards,
        collided,
    ):
        if cell in (
            (self.pick_pos.x, self.pick_pos.y),
            (self.drop_pos.x, self.drop_pos.y),
        ):
            return 0

        directions = {
            self.agents[i].direction
            for i in indices
        }

        if len(directions) < 2:
            return 0

        self.penalize(indices, rewards, collided)
        return 1
    
    def penalize(
        self,
        indices,
        rewards,
        collided,
    ):
        for index in indices:
            collided[index] = True
            rewards[index] -= 15.0

    # ---- Finish step ---
    def finish_step(
        self,
        rewards,
        delivered,
        collided,
        events,
    ):
        self.ticks += 1
        done = self.ticks >= self.max_ticks

        return (
            self.observe_all(),
            rewards,
            delivered,
            collided,
            events,
            done
        )

## 3. Q-Network and Replay Buffer

This section implements the core components of **Deep Q-Network (DQN)**.

### Q-Network
The **Q-Network** replaces the traditional Q-table with a neural network. It takes an agent's observation (`OBS_DIM = 10`) as input and predicts a Q-value for each possible action (`N_ACTIONS = 4`).

**Architecture:**
- **Input Layer:** 10 features (observation vector)
- **Hidden Layer 1:** 64 neurons + ReLU
- **Hidden Layer 2:** 64 neurons + ReLU
- **Output Layer:** 4 neurons (one Q-value per action)

**Flow:**

```text
Observation (10)
       │
       ▼
Linear (10 → 64)
       │
     ReLU
       │
Linear (64 → 64)
       │
     ReLU
       │
Linear (64 → 4)
       │
       ▼
Q-values
```

---

### Transition

A `Transition` stores one experience collected from the environment.

```python
(state, action, reward, next_state, done)
```

Where:
- **state** – Current observation.
- **action** – Action taken by the agent.
- **reward** – Reward received.
- **next_state** – Observation after taking the action.
- **done** – Whether the episode has ended.

---

### Replay Buffer

The **Replay Buffer** stores past experiences and randomly samples them for training. Random sampling reduces correlation between consecutive experiences, leading to more stable and efficient learning.

**Main Functions:**
- **`push()`** – Stores a new transition.
- **`sample()`** – Randomly selects a mini-batch and converts it into PyTorch tensors.
- **`__len__()`** – Returns the current number of stored transitions.

**Training Flow:**

```text
Environment
      │
      ▼
(state, action, reward, next_state, done)
      │
      ▼
Replay Buffer
      │
      ▼
Random Mini-Batch
      │
      ▼
PyTorch Tensors
      │
      ▼
Q-Network
```

In [9]:
class QNetwork(nn.Module):
    def __init__(self, obs_dim=OBS_DIM, n_actions=N_ACTIONS, hidden=64):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),  # 10 inputs - 64 outputs
            nn.Linear(hidden, hidden), nn.ReLU(),   # 64 inputs - 64 outputs
            nn.Linear(hidden, n_actions),           # 64 inputs - 4 outputs
        )
    def forward(self, observation):
        return self.network(observation)
    
Transition = namedtuple(
    "Transition",
    ["state", "action", "reward", "next_state", "done"],
)

class ReplayBuffer:
    def __init__(self, cap=100_000):
        self.buffer = deque(maxlen=cap)

    def push(self, *transition):
        self.buffer.append(Transition(*transition))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        return self.build_batch(batch)

    def __len__(self):
        return len(self.buffer)
    
    # -------------------------------------------

    def build_batch(self, batch):
        states = self.stack_states(batch)
        actions = self.actions_tensor(batch)
        rewards = self.rewards_tensor(batch)
        next_states = self.stack_next_states(batch)
        dones = self.done_tensor(batch)

        return (
            states,
            actions,
            rewards,
            next_states,
            dones,
        )
    
    def stack_states(self, batch):
        states = [item.state for item in batch]
        return torch.from_numpy(np.stack(states))
    
    def stack_next_states(self, batch):
        states = [item.next_state for item in batch]
        return torch.from_numpy(np.stack(states))
    
    def actions_tensor(self, batch):
        actions = [item.action for item in batch]
        return torch.tensor(actions, dtype=torch.long)
    
    def rewards_tensor(self, batch):
        rewards = [item.reward for item in batch]
        return torch.tensor(rewards, dtype=torch.float32)
    
    def done_tensor(self, batch):
        done = [item.done for item in batch]
        return torch.tensor(done, dtype=torch.float32)

In [ ]:
@dataclass
class TrainConfig:
    step_budget: int            = 1_500_000
    collision_budget: int       = 4_000
    walltime_s: float           = 10 * 60
    gamma: float                = 0.95
    learning_rate: float        = 1e-3
    batch_size: int             = 128
    buffer_size: int            = 100_000
    warmup: int                 = 2_000
    target_sync: int            = 1_000
    epsilon_start: float        = 1.0
    epsilon_end: float          = 0.05
    epsilon_decay_steps: int    = 120_000
    max_ticks_per_ep: int       = 120
    log_every: int              = 40_000


def train(config: TrainConfig = TrainConfig(), verbose=True):

    torch.set_num_threads(1)        # single-thread is fastest for tiny nets on CPU

#-------------------------------------------------------------------------------------------------------- Initialization Part -------------
    env = GridWorld(max_ticks=config.max_ticks_per_ep)              # Create Environment
    q_network = QNetwork().to(device)                               # Create Main Network
    target_network = QNetwork().to(device)                          # Create Target Network
    target_network.load_state_dict(q_network.state_dict())                       
    optim = torch.optim.Adam(                                       # Create Optimizer
        q_network.parameters(),
        lr=config.learning_rate
    )   
    buffer = ReplayBuffer(config.buffer_size)                       # Create Replay Buffer

    total_steps = 0
    total_collisions = 0
    grad_updates = 0
    ep = 0
    t0 = time.time()

    log = {                         # initialize logs
        "step": [],
        "ep": [],
        "avg_reward": [],
        "deliveries": [],
        "collisions": [],
        "eps": [],
    }

    ep_reward_window = deque(maxlen=50)
    ep_deliv_window = deque(maxlen=50)

    obs = env.reset()
    next_log = config.log_every

#------------------------------------------------------------------------------------------------- Training Part -------------------------------------  
    while True:
        # Loop breaking conditions
        if total_steps >= config.step_budget:
            reason = "step budget"
            break

        if total_collisions >= config.collision_budget:
            reason = "collision budget"
            break

        if (time.time() - t0) >= config.walltime_s:
            reason = "walltime"
            break
        
        epsilon = calculate_epsilon(config, total_steps)        # Epsilon Decay

        actions = choose_actions(obs, epsilon, q_network)                  # Chose actions

        obs2, rewards, delivered, collided, collision_events, done = env.step(actions)      # apply action

        store_transitions(
            buffer,
            obs,
            actions,
            rewards,
            obs2,
            done,
        )

        total_steps, total_collisions = update_statistics(
            rewards,
            collision_events,
            total_steps,
            total_collisions,
            ep_reward_window,
        )

        if len(buffer) >= config.warmup:
            train_step(
                q_network,
                target_network,
                buffer,
                optim,
                config,
            )

            grad_updates += 1

            if should_sync_target(grad_updates, config):
                sync_target_network(q_network, target_network)
            next_log = log_progress(
                verbose,
                total_steps,
                next_log,
                config,
                ep_reward_window,
                ep_deliv_window,
                total_collisions,
                ep,
                epsilon,
                t0,
                log,
            )
    
    print(f"Training stopped ({reason}). steps={total_steps} "
          f"collisions={total_collisions} time={time.time()-t0:.1f}s")
    return q_network, log 

#------------------------------------------------------------------------------------------------- All functions ------------------------------------

def calculate_epsilon(config, total_steps) -> float:
    fraction = min(1.0, total_steps / config.epsilon_decay_steps)
    return (config.epsilon_start + (config.epsilon_end - config.epsilon_start) * fraction)

def predict_q_values(observations, q_network):
    observations = torch.from_numpy(observations)
    with torch.no_grad():
        return q_network(observations).numpy()
        
def choose_action(q_values, epsilon):
    if random.random() < epsilon:
        return random.randrange(N_ACTIONS)
    return int(np.argmax(q_values))
    
def choose_actions(observations, epsilon, q_network):
    q_values = predict_q_values(observations, q_network)
    actions = []
    for values in q_values:
        actions.append(
            choose_action(values, epsilon)
        )
    return actions

def store_transitions(buffer, obs, actions, rewards, obs2, done):
    for i in range(N_AGENTS):
        buffer.push(
            obs[i],
            actions[i],
            rewards[i],
            obs2[i],
            float(done),
        )

def update_statistics(
    rewards,
    collision_events,
    total_steps,
    total_collisions,
    reward_window,
):
    total_steps += N_AGENTS
    total_collisions += collision_events
    reward_window.append(float(sum(rewards)))

    return total_steps, total_collisions

def sample_batch(buffer, config):
    return buffer.sample(config.batch_size)

def compute_target_q(
    target_network,
    next_state,
    reward,
    done,
    config,
):
    with torch.no_grad():
        q_next = target_network(next_state)
        q_next = q_next.max(dim=1).values

    return reward + config.gamma * (1.0 - done) * q_next

def predict_q(
    q_network,
    state,
    action,
):
    q_values = q_network(state)

    return (
        q_values
        .gather(1, action.unsqueeze(1))
        .squeeze(1)
    )

def optimize_network(
    q_network,
    optimizer,
    predicted_q,
    target_q,
):
    loss = F.smooth_l1_loss(
        predicted_q,
        target_q,
    )

    optimizer.zero_grad()

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        q_network.parameters(),
        5.0,
    )

    optimizer.step()

def train_step(
    q_network,
    target_network,
    buffer,
    optimizer,
    config,
):
    state, action, reward, next_state, done = sample_batch(
        buffer,
        config,
    )

    target_q = compute_target_q(
        target_network,
        next_state,
        reward,
        done,
        config,
    )

    predicted_q = predict_q(
        q_network,
        state,
        action,
    )

    optimize_network(
        q_network,
        optimizer,
        predicted_q,
        target_q,
    )

def sync_target_network(
    q_network,
    target_network,
):
    target_network.load_state_dict(
        q_network.state_dict()
    )

def should_sync_target(
    grad_updates,
    config,
):
    return (
        grad_updates %
        config.target_sync
        == 0
    )

def log_progress(
    verbose,
    total_steps,
    next_log,
    config,
    ep_reward_window,
    ep_deliv_window,
    total_collisions,
    ep,
    epsilon,
    t0,
    log,
):
    if not verbose:
        return next_log

    if total_steps < next_log:
        return next_log

    next_log += config.log_every

    average_reward = (
        float(np.mean(ep_reward_window))
        if ep_reward_window
        else 0.0
    )

    average_deliveries = (
        float(np.mean(ep_deliv_window))
        if ep_deliv_window
        else 0.0
    )

    print(
        f"[{int(time.time() - t0):4d}s] "
        f"steps={total_steps:>8d} "
        f"ep={ep:>4d} "
        f"eps={epsilon:.2f} "
        f"avg_tick_reward={average_reward:6.2f} "
        f"avg_deliveries/ep={average_deliveries:5.1f} "
        f"collisions={total_collisions}"
    )

    log["step"].append(total_steps)
    log["ep"].append(ep)
    log["avg_reward"].append(average_reward)
    log["deliveries"].append(average_deliveries)
    log["collisions"].append(total_collisions)
    log["eps"].append(epsilon)

    return next_log

In [25]:
qnet, log = train(TrainConfig())

Training stopped (collision budget). steps=140800 collisions=4000 time=124.5s
